In [1]:
import os
import re
import json
import time
import pandas as pd
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ===================== CONFIG =====================
IN_CSV   = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\uni_pre_cleaned.csv"
OUT_CSV  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\uni_yf_matches_from_openfigi.csv"

# --- OpenFIGI API KEY ---
# Preferred: set env var before running (PowerShell):  $env:OPENFIGI_API_KEY="YOUR_KEY"
OPENFIGI_API_KEY = os.getenv("OPENFIGI_API_KEY", "").strip()

# Caches
FIGI_CACHE_JSON = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\openfigi_cache.json"
YF_CACHE_JSON   = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\pre_data\yahoo_search_cache.json"

# Columns
NAME_COL    = "Name/Kind of Investment Item"
STOCKID_COL = "Stock ID"  # ISIN

# Concurrency / throttling
MAX_WORKERS      = 12
HTTP_PAUSE_SEC   = 0.18
HTTP_MAX_RETRIES = 2

# Bump to invalidate old Yahoo cache entries
CACHE_TAG = "yh_http_simple_v1"
def ck(q: str, region: str) -> str:
    return f"{CACHE_TAG}|{region}|{q.strip()}"

# Home exchange preferences by ISIN prefix
PREFS = {
    "CA": ["TO","V","NE","CN",""], "US": [""],
    "GB": ["L","IL"], "IE": ["IR"],
    "NL": ["AS"], "FR": ["PA"], "BE": ["BR"], "PT": ["LS"],
    "DE": ["DE","F","BE","DU","SG","MU","HM","HA"],
    "CH": ["SW"], "DK": ["CO"], "SE": ["ST"], "NO": ["OL"], "FI": ["HE"], "IS": ["IC"],
    "JP": ["T"], "CN": ["SS","SZ"], "HK": ["HK"], "TW": ["TW","TWO"],
    "SG": ["SI"], "AU": ["AX"], "NZ": ["NZ"],
    "IN": ["NS","BO"], "ES": ["MC","BC","BI","MA"],
    "IT": ["MI"], "AT": ["VI"], "TR": ["IS"], "GR": ["AT"], "IL": ["TA"],
    "ZA": ["JO"], "BR": ["SA"], "MX": ["MX"], "AR": ["BA"], "CL": ["SN"], "PE": ["LM"], "CO": ["COLO"],
    "KR": ["KS","KQ"], "TH": ["BK"], "MY": ["KL"], "ID": ["JK"], "PH": ["PS"],
    "VN": ["VN","HN"], "AE": ["AD","DU"], "QA": ["QA"], "SA": ["SR"], "BH": ["BH"], "KW": ["KW"], "OM": ["MS"],
}

# Yahoo "region" by ISIN prefix (keeps results focused)
REGION_BY_PREFIX = {
    "US":"US","CA":"CA","GB":"GB","IE":"GB","FR":"FR","DE":"DE","NL":"NL","BE":"BE","PT":"PT",
    "ES":"ES","IT":"IT","CH":"CH","DK":"DK","SE":"SE","NO":"NO","FI":"FI",
    "JP":"JP","HK":"HK","CN":"HK","TW":"TW","SG":"SG","AU":"AU","NZ":"NZ","IN":"IN","KR":"KR",
    "BR":"BR","MX":"MX","AR":"AR",
}
def region_for_prefix(pref: str) -> str:
    return REGION_BY_PREFIX.get(pref.upper(), "US")

# Yahoo HTTP search
YF_SEARCH_URL = "https://query2.finance.yahoo.com/v1/finance/search"
YF_HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/124.0.0.0 Safari/537.36"),
    "Accept": "application/json,text/plain,*/*",
    "Connection": "keep-alive",
}
YF_DEFAULT_PARAMS = {"lang":"en-US","region":"US","quotesCount":20,"newsCount":0}

# ===================== UTILS =====================
def load_json(path):
    p = Path(path)
    if not p.exists(): return {}
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}

def save_json(obj, path):
    try:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(obj, f, ensure_ascii=False)
    except Exception:
        pass

def isin_prefix(isin: str) -> str:
    return (isin[:2].upper() if isinstance(isin, str) and len(isin) >= 2 else "")

def yf_suffix(symbol: str) -> str:
    return symbol.split(".")[-1].upper() if "." in symbol else ""

def normalized(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^A-Za-z0-9\s]", " ", (s or "").strip())).strip().lower()

# ===================== OPENFIGI (name only) =====================
def openfigi_batch_lookup(isins, api_key: str):
    url = "https://api.openfigi.com/v3/mapping"
    headers = {"Content-Type": "application/json"}
    if api_key:
        headers["X-OPENFIGI-APIKEY"] = api_key

    out = {}
    for i in range(0, len(isins), 100):
        chunk = isins[i:i+100]
        payload = [{"idType": "ID_ISIN", "idValue": x} for x in chunk]
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=20)
            if resp.status_code != 200:
                time.sleep(0.5)
                for req in chunk:
                    out.setdefault(req, {"name": ""})
                continue
            arr = resp.json()
            for req, res in zip(chunk, arr):
                if res and "data" in res and res["data"]:
                    d0 = res["data"][0]
                    out[req] = {"name": d0.get("name", "") or ""}
                else:
                    out[req] = {"name": ""}
            time.sleep(0.12)
        except Exception:
            time.sleep(0.5)
            for req in chunk:
                out.setdefault(req, {"name": ""})
    return out

def resolve_openfigi_names(isins, api_key, cache_path):
    cache = load_json(cache_path)
    need = [i for i in isins if i not in cache]
    print(f"OpenFIGI: need {len(need)} of {len(isins)} ISINs (cached: {len(isins)-len(need)})")
    if need:
        resolved = openfigi_batch_lookup(need, api_key)
        cache.update(resolved)
        save_json(cache, cache_path)
    return {i: cache.get(i, {"name": ""}) for i in isins}

# ===================== YAHOO VIA HTTP =====================
_session = requests.Session()
_session.headers.update(YF_HEADERS)

def yahoo_http_search(query: str, region: str = "US"):
    if not query:
        return []
    for attempt in range(HTTP_MAX_RETRIES + 1):
        try:
            params = {**YF_DEFAULT_PARAMS, "q": query, "region": region}
            r = _session.get(YF_SEARCH_URL, params=params, timeout=15)
            if r.status_code == 200:
                data = r.json()
                time.sleep(HTTP_PAUSE_SEC)
                return data.get("quotes", []) or []
            time.sleep(HTTP_PAUSE_SEC + 0.2 * (attempt + 1))
        except Exception:
            time.sleep(HTTP_PAUSE_SEC + 0.3 * (attempt + 1))
    return []

OTC_EXCH = {"PNK","OBB","OOTC"}  # Yahoo 'exchange' codes for OTC/PK

def rank_yf_candidate(hit, want_name_norm, isin_pref):
    """Simple scoring + strong home-exchange preference + OTC penalty."""
    longname = (hit.get("longname") or hit.get("shortname") or "")
    symbol   = (hit.get("symbol") or "").strip()
    qt       = (hit.get("quoteType") or "").lower()
    exch     = (hit.get("exchange") or "").upper()

    score = 0
    nn = normalized(longname)

    # Name similarity
    if nn == want_name_norm: score += 8
    elif want_name_norm and (nn.startswith(want_name_norm) or want_name_norm.startswith(nn)): score += 5
    elif want_name_norm and want_name_norm in nn: score += 2

    if qt in {"equity","etf"}: score += 1

    # Home suffix preference
    if isin_pref in PREFS and symbol:
        suf = yf_suffix(symbol)
        prefs = PREFS[isin_pref]
        if suf in prefs:
            score += (12 + len(prefs) - prefs.index(suf))  # strong boost
        else:
            score -= 2

    # Penalize OTC/PK for non-US ISINs
    if isin_pref != "US" and exch in OTC_EXCH:
        score -= 6

    return score, symbol, longname

def best_for_query(q: str, isin_pref: str, region: str):
    quotes = yahoo_http_search(q, region=region)
    want = normalized(q)
    ranked, seen = [], set()
    home_suffixes = set(PREFS.get(isin_pref, []))
    for h in quotes:
        sc, sym, mname = rank_yf_candidate(h, want_name_norm=want, isin_pref=isin_pref)
        if not sym or sym in seen:
            continue
        seen.add(sym)
        ranked.append((sc, sym, mname, yf_suffix(sym) in home_suffixes))
    # If any home-exchange candidate exists, keep only those
    if any(c[3] for c in ranked):
        ranked = [c for c in ranked if c[3]]
    ranked.sort(key=lambda x: x[0], reverse=True)
    return ranked[0] if ranked else None  # (score, symbol, match_name, in_home)

def resolve_yahoo_symbols_http(query_tuples, cache_path):
    """
    query_tuples: list of (query_string, isin_prefix, region, source_label)
    Returns dict[(query, region)] -> {"symbol","match","score"}
    """
    cache = load_json(cache_path)

    def is_bad(rec):
        return (not rec) or (not rec.get("symbol")) or (rec.get("score", -1) < 0)

    to_query = [(q, p, r, s) for (q, p, r, s) in query_tuples if q and (ck(q, r) not in cache or is_bad(cache.get(ck(q, r))))]
    print(f"Yahoo: need {len(to_query)} of {len(query_tuples)} queries (cached: {len(query_tuples)-len(to_query)})")

    def worker(item):
        q, p, r, s = item
        best = best_for_query(q, p, r)
        if best:
            sc, sym, mname, _ = best
            rec = {"symbol": sym, "match": mname, "score": sc}
        else:
            rec = {"symbol":"", "match":"", "score": -1}
        return q, r, rec

    if to_query:
        results = {}
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = {ex.submit(worker, item): item for item in to_query}
            done = 0; total = len(to_query)
            for fut in as_completed(futs):
                q, r, rec = fut.result()
                results[ck(q, r)] = rec
                done += 1
                if done % 100 == 0 or done == total:
                    print(f"Yahoo: {done}/{total} queries completed")
        cache.update(results)
        save_json(cache, cache_path)

    out = {}
    for q, p, r, s in query_tuples:
        out[(q, r)] = cache.get(ck(q, r), {"symbol": "", "match": "", "score": -1})
    return out

# ===================== MAIN =====================
def main():
    if not OPENFIGI_API_KEY:
        print("⚠️  Set OPENFIGI_API_KEY (env var or in the script) for best performance.")

    df = pd.read_csv(IN_CSV, dtype=str, keep_default_na=False, na_values=[], encoding="cp1252")

    # rows with ISIN
    work_mask = df.get(STOCKID_COL, pd.Series([""]*len(df))).astype(str).str.strip().ne("")
    df_work = df.loc[work_mask, [NAME_COL, STOCKID_COL]].copy()

    # 1) ISIN -> OpenFIGI name (cached)
    isins = df_work[STOCKID_COL].astype(str).str.strip()
    isins = isins[isins.ne("").dropna()].drop_duplicates().tolist()
    figi_map = resolve_openfigi_names(isins, OPENFIGI_API_KEY, FIGI_CACHE_JSON)
    df_work["OpenFIGI Company Name"] = df_work[STOCKID_COL].map(lambda i: figi_map.get(i, {}).get("name", ""))

    # 2) Build per-row queries: FIGI name and CSV name ONLY (no variants)
    global_queries = []   # (query, pref, region, src)
    query_set = set()
    row_qs = {}           # idx -> list[(q, region, src)]

    for idx, row in df_work.iterrows():
        isin = str(row[STOCKID_COL]).strip()
        pref = isin_prefix(isin)
        region = region_for_prefix(pref)

        names = []
        figi_name = str(row.get("OpenFIGI Company Name", "")).strip()
        csv_name  = str(row[NAME_COL]).strip()

        if figi_name:
            names.append(("openfigi", figi_name))
        if csv_name and csv_name != figi_name:
            names.append(("csv", csv_name))

        pairs = []
        for src, q in names:
            if (q, region) not in query_set:
                query_set.add((q, region))
                global_queries.append((q, pref, region, src))
            pairs.append((q, region, src))
        row_qs[idx] = pairs

    # 3) Resolve Yahoo (cached; refresh blanks)
    results = resolve_yahoo_symbols_http(global_queries, YF_CACHE_JSON)

    # 4) Pick best per row (prefer home exchange if present among candidates)
    df_work["Yahoo ticker"] = ""
    df_work["Matching Name"] = ""
    df_work["Match Source"] = ""

    for idx, pairs in row_qs.items():
        pref = isin_prefix(str(df_work.at[idx, STOCKID_COL]).strip())
        home = set(PREFS.get(pref, []))
        cands = []
        for q, region, src in pairs:
            rec = results.get((q, region), {})
            sym  = rec.get("symbol","")
            name = rec.get("match","")
            sc   = rec.get("score",-1)
            if sym:
                cands.append((sc, sym, name, src, yf_suffix(sym) in home))
        if not cands:
            continue
        # if any home exchange exists, keep only those
        if any(c[-1] for c in cands):
            cands = [c for c in cands if c[-1]]
        cands.sort(key=lambda x: x[0], reverse=True)
        best = cands[0]
        df_work.at[idx, "Yahoo ticker"]  = best[1]
        df_work.at[idx, "Matching Name"] = best[2]
        df_work.at[idx, "Match Source"]  = best[3]

    # 5) Output + summary
    out = df_work[[NAME_COL, "OpenFIGI Company Name", STOCKID_COL, "Yahoo ticker", "Matching Name", "Match Source"]].copy()
    out.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"✅ Saved: {OUT_CSV}")

    processed = len(df_work)
    matched   = int(out["Yahoo ticker"].astype(str).str.strip().ne("").sum())
    unmatched = processed - matched
    matched_by = out.loc[out["Yahoo ticker"].astype(str).str.strip().ne(""), "Match Source"].value_counts().to_dict()

    print("---- Summary ----")
    print(f"Rows with non-empty Stock ID (processed): {processed}")
    print(f"Matched via OpenFIGI name:               {matched_by.get('openfigi', 0)}")
    print(f"Matched via CSV name:                    {matched_by.get('csv', 0)}")
    print(f"TOTAL matched:                           {matched}")
    print(f"Unmatched:                               {unmatched}")
    rate = (matched/processed*100) if processed else 0.0
    print(f"Hit rate:                                {rate:.2f}%")

if __name__ == "__main__":
    main()


⚠️  Set OPENFIGI_API_KEY (env var or in the script) for best performance.
OpenFIGI: need 0 of 3234 ISINs (cached: 3234)
Yahoo: need 5111 of 5111 queries (cached: 0)
Yahoo: 100/5111 queries completed
Yahoo: 200/5111 queries completed
Yahoo: 300/5111 queries completed
Yahoo: 400/5111 queries completed
Yahoo: 500/5111 queries completed
Yahoo: 600/5111 queries completed
Yahoo: 700/5111 queries completed
Yahoo: 800/5111 queries completed
Yahoo: 900/5111 queries completed
Yahoo: 1000/5111 queries completed
Yahoo: 1100/5111 queries completed
Yahoo: 1200/5111 queries completed
Yahoo: 1300/5111 queries completed
Yahoo: 1400/5111 queries completed
Yahoo: 1500/5111 queries completed
Yahoo: 1600/5111 queries completed
Yahoo: 1700/5111 queries completed
Yahoo: 1800/5111 queries completed
Yahoo: 1900/5111 queries completed
Yahoo: 2000/5111 queries completed
Yahoo: 2100/5111 queries completed
Yahoo: 2200/5111 queries completed
Yahoo: 2300/5111 queries completed
Yahoo: 2400/5111 queries completed
Yaho

In [ ]:
import yfinance as yf

ticker_yahoo = yf.Ticker("CA1254911003")
print(ticker_yahoo.ticker)

MAERSK-A.CO


In [2]:
import requests
r = requests.get(
    "https://query2.finance.yahoo.com/v1/finance/search",
    params={"q":"Apple", "lang":"en-US","region":"US","quotesCount":5,"newsCount":0},
    headers={"User-Agent":"Mozilla/5.0"}
)
print(r.status_code, r.text[:2000])


200 {"explains":[],"count":5,"quotes":[{"exchange":"NMS","shortname":"Apple Inc.","quoteType":"EQUITY","symbol":"AAPL","index":"quotes","score":37700.0,"typeDisp":"Equity","longname":"Apple Inc.","exchDisp":"NASDAQ","sector":"Technology","sectorDisp":"Technology","industry":"Consumer Electronics","industryDisp":"Consumer Electronics","dispSecIndFlag":true,"isYahooFinance":true},{"exchange":"NEO","shortname":"APPLE CDR (CAD HEDGED)","quoteType":"EQUITY","symbol":"AAPL.NE","index":"quotes","score":20087.0,"typeDisp":"Equity","longname":"Apple Inc.","exchDisp":"NEO","sector":"Technology","sectorDisp":"Technology","industry":"Consumer Electronics","industryDisp":"Consumer Electronics","isYahooFinance":true},{"exchange":"NYQ","shortname":"Apple Hospitality REIT, Inc.","quoteType":"EQUITY","symbol":"APLE","index":"quotes","score":20038.0,"typeDisp":"Equity","longname":"Apple Hospitality REIT, Inc.","exchDisp":"NYSE","sector":"Real Estate","sectorDisp":"Real Estate","industry":"REIT—Hotel & M